In [33]:
import pandas as pd
import numpy as np


In [34]:
df = pd.read_csv("../data/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
df

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,8.0
1,0.0,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,8.0
2,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,10.0,0.0,1.0,13.0,6.0,8.0
3,0.0,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,1.0,11.0,6.0,8.0
4,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,8.0,5.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70687,1.0,0.0,1.0,1.0,37.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,0.0,0.0,6.0,4.0,1.0
70688,1.0,0.0,1.0,1.0,29.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,2.0,0.0,0.0,1.0,1.0,10.0,3.0,6.0
70689,1.0,1.0,1.0,1.0,25.0,0.0,0.0,1.0,0.0,1.0,...,1.0,0.0,5.0,15.0,0.0,1.0,0.0,13.0,6.0,4.0
70690,1.0,1.0,1.0,1.0,18.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0


Okay, so the data has largely been pre-cleaned for us, there are no missing values...

Features:
Binary (14):
* HighBP
* HighChol
* CholCheck (0=no cholesterol check in last 5 years)
* Smoker (1=has smoked at least 100 cigs in their entire life)
* Stroke (0=has never had a stroke)
* HeartDiseaseorAttack (0=has never had coronary heart disease or myocardial infarcion)
* PhysActivity (1=has done physical activity in the last 30 days, not including their job)
* Fruits (1=consumes fruits 1+ times a day)
* Veggies (1=consumes vegetables 1+ times a day)
* HvyAlcoholConsump (if man, 1=more than 14 drinks per week, if woman, 1=more than 7 drinks per week)
* AnyHealthcare (1=has healthcare coverage)
* NoDocbcCost (1=there has been a time in the last 12 months when they needed, but could not see a dr bc of cost)
* DiffWalk (1=has serious difficulty walking or climbing stairs)
* Sex (0=female, 1=male)

Quantitative (3):
* BMI
* MentHlth (numb of days over last 30 days with bad mental health)
* PhysHlth (numb of days over last 30 days with bad phys health)

Ordinal (4):
* GenHlth (Would you say that in general your health is: scale 1-5 1 = excellent 2 = very good 3 = good 4 = fair 5 = poor)
* Age (1=[18,24], 2=[25,29], 3=[30,34], 4=[35,39], 5=[40,44], 6=[45,49], 7=[50,54], 8=[55,59], 9=[60,64], 10=[65,69], 11=[70,74], 12=[75,79], 13=[80,99])
* Education (1 = Never attended school or only kindergarten 2 = Grades 1 through 8 (Elementary) 3 = Grades 9 through 11 (Some high school) 4 = Grade 12 or GED (High school graduate) 5 = College 1 year to 3 years (Some college or technical school) 6 = College 4 years or more (College graduate))
* Income (1=[0,10000), 2=[10000,15000), 3=[15000,20000), 4=[20000,25000), 5=[25000,35000), 6=[35000,50000), 7=[50000,75000), 8=75000+)

For age and income, it makes most sense to use midpoint values to change from ordinal values to quantitative values. We'll then standardize all our data, before splitting our dataframe into training, validation and testing datasets.

In [35]:
age_map = {1: 21, 2: 27, 3: 32, 4: 37, 5: 42, 6: 47, 7: 52, 8: 57, 9: 62, 10: 67, 11: 72, 12: 77, 13: 85}
income_map = {1: 5000, 2: 12500, 3: 17500, 4: 22500, 5: 30000, 6: 42500, 7: 62500, 8: 87500}
df["IncomeMid"] = df["Income"].map(income_map)
df["AgeMid"] = df["Age"].map(age_map)

# Standardizing
for category in ["BMI", "MentHlth", "PhysHlth", "GenHlth", "Education", "IncomeMid", "AgeMid", ]:
    mean = df[category].mean()
    std = df[category].std()
    df[category] = (df[category] - mean) / std

# Beginning by splitting the dataframe into three parts, using proportions 8:2:2 for training:validation:testing

np.random.seed(42)
shuffled_df = df.sample(frac=1).reset_index(drop=True)
shuffled_df = shuffled_df.drop(columns=["Income","Age"])
train_df = shuffled_df.iloc[:(int(len(shuffled_df)*(8/12)))]
val_df = shuffled_df.iloc[int(len(shuffled_df)*(8/12)):int(len(shuffled_df)*(10/12))]
test_df = shuffled_df.iloc[int(len(shuffled_df)*(10/12)):]

# Separating Xs and Ys
X_train = train_df.drop(columns=["Diabetes_binary"])
X_train = X_train.values.T
Y_train = train_df["Diabetes_binary"]
Y_train = np.asarray(Y_train).reshape(-1,1)
X_val = val_df.drop(columns=["Diabetes_binary"])
X_val = X_val.values.T
Y_val = val_df["Diabetes_binary"]
Y_val = np.asarray(Y_val).reshape(-1,1)
X_test = test_df.drop(columns=["Diabetes_binary"])
X_test = X_test.values.T
Y_test = test_df["Diabetes_binary"]
Y_test = np.asarray(Y_test).reshape(-1,1)

In [36]:
input_size = X_train.shape[0]
output_size = 1
hidden_layer_size = 15

### Activation functions:

In [37]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
def reLu(x):
    return np.maximum(0, x)